# 05 Final Load Prep

Compute final KPIs, prepare the Tableau-ready dataset, and export the exact files used in the dashboard.

**Outputs produced:**
| File | Purpose |
|---|---|
| `data/processed/tableau_ready_dataset.csv` | Full transaction-level dataset for Tableau |
| `data/processed/kpi_summary.csv` | High-level KPI snapshot card |
| `data/processed/sales_by_category.csv` | Revenue & transactions grouped by category |
| `data/processed/sales_by_payment.csv` | Revenue by payment method |
| `data/processed/sales_by_location.csv` | Online vs In-store breakdown |
| `data/processed/discount_impact.csv` | Revenue comparison — discounted vs non-discounted |
| `data/processed/sales_monthly_trend.csv` | Monthly revenue trend |
| `data/processed/sales_by_item.csv` | Top items by revenue |

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import compute_kpis

In [ ]:
DATA_PATH          = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'
PROCESSED_DIR      = PROJECT_ROOT / 'data/processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=['transaction_date'])
print(f'Loaded cleaned dataset: {df.shape}')
df.head(3)

## 5.1 Ensure All Derived Columns Are Present

In [ ]:
from scripts.etl_pipeline import add_derived_features
# Calling this just guarantees the columns exist if Notebook 02 was bypassed
df = add_derived_features(df)

print('All derived columns confirmed.')
print(df.columns.tolist())

## 5.2 Core KPI Summary

In [ ]:
kpis = compute_kpis(df)

print('='*55)
print('          RETAIL STORE SALES — KPI SUMMARY')
print('='*55)
for k, v in kpis.items():
    label = k.replace('_', ' ').title()
    if isinstance(v, float):
        print(f'  {label:<35}: {v:,.2f}')
    elif isinstance(v, int):
        print(f'  {label:<35}: {v:,}')
    else:
        print(f'  {label:<35}: {v}')
print('='*55)

kpi_df = pd.DataFrame([{'kpi': k, 'value': str(v)} for k, v in kpis.items()])
kpi_df.to_csv(PROCESSED_DIR / 'kpi_summary.csv', index=False)
print(f'\nSaved: kpi_summary.csv')

## 5.3 Aggregated Tableau Exports

In [ ]:
def agg_sales(group_col: str) -> pd.DataFrame:
    return (
        df.groupby(group_col, observed=True)
        .agg(
            total_revenue=('total_spent', 'sum'),
            total_transactions=('transaction_id', 'count'),
            avg_order_value=('total_spent', 'mean'),
            total_quantity=('quantity', 'sum')
        )
        .reset_index()
        .round(2)
    )

# 1. Sales by Category
sales_by_cat = agg_sales('category').sort_values('total_revenue', ascending=False)
sales_by_cat.to_csv(PROCESSED_DIR / 'sales_by_category.csv', index=False)

# 2. Sales by Payment Method
sales_by_pay = agg_sales('payment_method').sort_values('total_revenue', ascending=False)
sales_by_pay.to_csv(PROCESSED_DIR / 'sales_by_payment.csv', index=False)

# 3. Sales by Location
sales_by_loc = agg_sales('location').sort_values('total_revenue', ascending=False)
sales_by_loc.to_csv(PROCESSED_DIR / 'sales_by_location.csv', index=False)

# 4. Discount Impact
discount_impact = agg_sales('discount_applied')
discount_impact.to_csv(PROCESSED_DIR / 'discount_impact.csv', index=False)

# 5. Sales Monthly Trend
monthly_trend = (
    df.groupby(['transaction_year', 'transaction_month'])
    .agg(total_revenue=('total_spent', 'sum'), total_transactions=('transaction_id', 'count'))
    .reset_index()
    .sort_values(['transaction_year', 'transaction_month'])
)
monthly_trend.to_csv(PROCESSED_DIR / 'sales_monthly_trend.csv', index=False)

# 6. Sales by Item
sales_by_item = agg_sales('item').sort_values('total_revenue', ascending=False)
sales_by_item.to_csv(PROCESSED_DIR / 'sales_by_item.csv', index=False)

print('All aggregated files saved successfully.')

## 5.4 Export Tableau-Ready Transaction Dataset

In [ ]:
# The full cleaned dataset is essentially our Tableau-ready dataset now.
# Let's just ensure it's saved to the proper location.
tableau_df = df.copy()

tableau_df.to_csv(TABLEAU_READY_PATH, index=False)
print(f'Saved Tableau-ready dataset to {TABLEAU_READY_PATH}')
print(f'Final shape: {tableau_df.shape}')
tableau_df.head(5)

## 5.5 Final Checklist

In [ ]:
files_to_check = [
    'cleaned_dataset.csv',
    'tableau_ready_dataset.csv',
    'kpi_summary.csv',
    'sales_by_category.csv',
    'sales_by_payment.csv',
    'sales_by_location.csv',
    'discount_impact.csv',
    'sales_monthly_trend.csv',
    'sales_by_item.csv',
]

print('Pipeline output checklist:')
print('-' * 55)
for fname in files_to_check:
    fpath = PROCESSED_DIR / fname
    status = '✅' if fpath.exists() else '❌ MISSING'
    size   = f'{fpath.stat().st_size / 1024:.1f} KB' if fpath.exists() else '—'
    print(f'  {status}  {fname:<45} {size}')

print()
print('All outputs ready for Tableau. Load tableau_ready_dataset.csv as the primary data source.')

## KPI Reference Card

| KPI | Definition | Dashboard Use |
|---|---|---|
| **Total Transactions** | Count of all rows | Executive KPI card |
| **Total Revenue** | Sum of `total_spent` | Executive KPI card |
| **Avg Order Value (AOV)** | Mean of `total_spent` | Comparison baseline |
| **Discount Rate** | % of orders where `discount_applied = 1` | Discount monitoring |
| **Online vs In-store** | % split of sales by `location` | Sales channel performance |
| **Revenue Bucket** | Low/Medium/High spend tiers | Customer segmentation |

**Analytical pipeline complete.** Upload `tableau_ready_dataset.csv` to Tableau Public to build the dashboard.